## 0. Введение

Этот ноутбук демонстрирует базовый запуск обучения корректора `CharLM` в версии `0.1.12` и выше. В примере показано, как использовать встроенные средства библиотеки `manuscript-ocr` для обучения собственной модели коррекции OCR-ошибок на пользовательских данных.

Для обучения корректора в примере используются:

- `words_path` — список слов;
- `text_path` — текстовый корпус;
- `pairs_path` — пары ошибочного и корректного написания;
- `charset_path` — файл допустимого набора символов.

Подробная информация о формате входных данных и параметрах обучения приведена в полной документации проекта: https://konstantinkozhin.github.io/manuscript-ocr/0.1.12/en/api/correctors.html

## 1. Установка зависимостей

Для загрузки примерных данных в данном ноутбуке дополнительно используется библиотека `huggingface_hub`.

Она не является частью библиотеки `manuscript-ocr`, не входит в обязательный набор зависимостей проекта и используется только для автоматического скачивания файлов датасета в рамках данного примера.

In [ ]:
%pip install "manuscript-ocr[dev]>=0.1.12"

In [ ]:
%pip install huggingface_hub

## 2. Скачивание данных

In [1]:
from pathlib import Path
from huggingface_hub import hf_hub_download


repo_id = "anna4uonline/TextCorrectorDataset"
local_dir = Path("datasets") / "TextCorrectorDataset" / "prereform_russian_data"
local_dir.mkdir(parents=True, exist_ok=True)

files = [
    "prereform_russian_data/charset.txt",
    "prereform_russian_data/pred_StackMix-YeniseiGovReports.csv",
    "prereform_russian_data/prereform_texts.txt",
    "prereform_russian_data/prereform_words.txt",
]

downloaded = {}
for file in files:
    path = hf_hub_download(
        repo_id=repo_id,
        repo_type="dataset",
        filename=file,
        local_dir=local_dir.parent,
        local_dir_use_symlinks=False,
    )
    downloaded[Path(file).name] = str(Path(path).resolve())

charset_path = downloaded["charset.txt"]
pairs_path = downloaded["pred_StackMix-YeniseiGovReports.csv"]
text_path = downloaded["prereform_texts.txt"]
words_path = downloaded["prereform_words.txt"]

print("charset_path =", charset_path)
print("pairs_path   =", pairs_path)
print("text_path    =", text_path)
print("words_path   =", words_path)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:206: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


charset.txt:   0%|          | 0.00/694 [00:00<?, ?B/s]

prereform_russian_data/pred_StackMix-Yen(…):   0%|          | 0.00/89.8M [00:00<?, ?B/s]

prereform_russian_data/prereform_texts.t(…):   0%|          | 0.00/70.9M [00:00<?, ?B/s]

prereform_russian_data/prereform_words.t(…):   0%|          | 0.00/36.3M [00:00<?, ?B/s]

charset_path = /content/datasets/TextCorrectorDataset/prereform_russian_data/charset.txt
pairs_path   = /content/datasets/TextCorrectorDataset/prereform_russian_data/pred_StackMix-YeniseiGovReports.csv
text_path    = /content/datasets/TextCorrectorDataset/prereform_russian_data/prereform_texts.txt
words_path   = /content/datasets/TextCorrectorDataset/prereform_russian_data/prereform_words.txt


## 3. Запуск обучения

In [2]:
from manuscript.correctors import CharLM

CharLM.train(
    words_path=words_path,
    text_path=text_path,
    pairs_path=pairs_path,
    charset_path=charset_path,
    epochs=1,
)

Device: cuda
Words: 1,500,000
Train pairs: 283646, Eval pairs: 2865
Building substitutions dictionary...
Substitutions: 1090, saved to exp_charlm/substitutions.json
  л→м: 11652
  ъ→ѣ: 7160
  о→а: 6041
  ь→я: 5778
  ь→ъ: 5728
  п→т: 4815
  е→ѣ: 4293
  й→я: 3954
  п→н: 3850
  ы→ь: 3811
NgramDataset: /content/datasets/TextCorrectorDataset/prereform_russian_data/prereform_texts.txt
PairsDataset: 283646 pairs
MixedDataset: pairs_ratio=0.80 (80% OCR)


/usr/local/lib/python3.12/dist-packages/manuscript/correctors/_charlm/model.py:16: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(enc_layer, num_layers=n_layers)
/usr/local/lib/python3.12/dist-packages/manuscript/correctors/_charlm/train.py:177: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler() if cfg.get("use_amp", False) and device == "cuda" else None


Using mixed precision (AMP)


Epoch 1:   0%|          | 0/5860 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/manuscript/correctors/_charlm/train.py:210: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Epoch 1: 100%|██████████| 5860/5860 [05:38<00:00, 17.29it/s, acc=62.3%, loss=0.618]


[Epoch 1] loss=0.8283 acc=49.70%
---- MLM examples ----
INPUT   : <MASK>осова
TARGET  : Носова
PREDICT : носова

INPUT   : закр<MASK>пощать
TARGET  : закрѣпощать
PREDICT : закрѣпощать

INPUT   : дн<MASK><MASK>тъ
TARGET  : днюетъ
PREDICT : дняетъ



...

/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...


Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/__init__.py", line 120, in call
    converted_proto = _c_api_utils.call_onnx_api(
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
             ^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/__init__.py", line 115, in _partial_convert_version
    return onnx.version_converter.convert_version(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnx/version_converter.py", line 39, in convert_version
    converted_model_str = C.convert_version(model_str, target_version)
                          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: /github/workspace/onnx/version_converter/adapters/no_previous_version.h:24: adapt: Assertion `

[torch.onnx] Translate the graph into ONNX... ✅
Applied 22 of general pattern rewrite rules.
[OK] ONNX model saved to: exp_charlm/charlm_epoch_1.onnx
Simplifying ONNX model...
[OK] Model simplified successfully

Export complete: exp_charlm/charlm_epoch_1.onnx
ONNX model exported successfully: exp_charlm/charlm_epoch_1.onnx


'exp_charlm/checkpoints/charlm_epoch_1.pt'

По окончании обучения лучшая по точности модель автоматически конвертируется в `ONNX` формат.

## Применение экспортированной модели

После обучения и экспорта модель корректора можно сразу использовать для постобработки текста. В данном примере создается объект `Page` с тестовой строкой, после чего к нему применяется экспортированная модель `CharLM`.

In [17]:
from pathlib import Path

from manuscript.correctors import CharLM
from manuscript.utils import create_page_from_text

exp_dir = Path("exp_charlm")

page = create_page_from_text(["закрѣпощать"])

corrector = CharLM(
    weights=str(exp_dir / "charlm_epoch_1.onnx"),
    vocab=str(exp_dir / "vocab.json"),
)

result_page = corrector.predict(page)

def page_to_text(page):
    lines = []
    for block in page.blocks:
        for line in block.lines:
            texts = [span.text for span in line.text_spans if span.text]
            if texts:
                lines.append(" ".join(texts))
    return "\n".join(lines)

print("До коррекции:")
print(page_to_text(page))

print("После коррекции:")
print(page_to_text(result_page))


[CharLM] Device configuration:
  Requested device: cpu
  Requested providers: ['CPUExecutionProvider']
  Active providers: ['CPUExecutionProvider']
  Running on: CPUExecutionProvider
До коррекции:
закрѣпощать
После коррекции:
закрѣпощать


Следует учитывать, что приведенный выше результат носит демонстрационный характер, поскольку модель в данном примере обучалась только `1` эпоху.

Таким образом, данный пример показывает, что `manuscript-ocr` предоставляет возможность обучать, экспортировать и затем использовать собственные модели коррекции OCR-текста на пользовательских данных.